# CocoaGuard AI — classifier training

Fine-tunes a **MobileNetV3-small** on Ghana cocoa images to classify:

`anthracnose | black_pod | cssvd | healthy`

**Critical contract with the app (`lib/model.ts` + `lib/classifier.ts`):**

1. A `Rescaling(1./127.5, offset=-1)` layer is baked as the **first** layer of the model, so the browser only has to do `fromPixels -> resizeBilinear(224) -> predict` on raw `[0,255]` pixels. No preprocessing in JS.
2. Class folder names are exactly the 4 slugs above. `flow_from_directory` sorts folder names alphabetically, so the output index order is `['anthracnose', 'black_pod', 'cssvd', 'healthy']` — this **must** match `CLASS_LABELS` in `lib/model.ts`.
3. Export with `tensorflowjs_converter --quantize_float16` -> copy `web_model.zip` contents into `public/models/` (model.json + .bin).

Run all cells top-to-bottom. It takes ~1-2h on the free Colab GPU.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow tensorflowjs dataset-tools kaggle

## 1. Download data

**Primary — KaraAgroAI Cocoa** (CC0, Harvard Dataverse DOI `10.7910/DVN/BBGQSP`, no login needed). Detection format: images + annotation JSON. Gives us `cssvd`, `anthracnose`, `healthy`.

**Black pod — Kaggle `zaldyjr/cacao-diseases`** (needs a `kaggle.json` uploaded to Colab). If you skipped Kaggle, fall back to the GitHub mirror `Br-Al/Cocoa-diseases` (no auth).

In [ ]:
!mkdir -p karaagro
!wget -q --show-progress "https://dataverse.harvard.edu/api/access/dataset/:persistentId?persistentId=doi:10.7910/DVN/BBGQSP" -O karaagro.zip || true
!unzip -q -o karaagro.zip -d karaagro 2>/dev/null || echo "zip failed — check the downloaded file layout"

# The archive may extract with a nested folder; flatten one level for convenience.
!find karaagro -type d -mindepth 2 -maxdepth 3 | head -20
!find karaagro -iname '*.json' | head -20
!find karaagro -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' | wc -l

In [ ]:
# Upload kaggle.json once (Settings -> Secrets, or files.upload below), then:
import os
if os.path.exists('/content/drive/MyDrive/kaggle.json'):
    !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d zaldyjr/cacao-diseases -p kaggle_cacao --unzip 2>/dev/null || echo "Kaggle failed — use the GitHub mirror:"
!git clone --depth 1 https://github.com/Br-Al/Cocoa-diseases.git kaggle_cacao 2>/dev/null || echo "Mirror clone failed"

!find kaggle_cacao -maxdepth 3 -type d | head -30

## 2. Build per-class image folders

KaraAgroAI is detection format, so we **crop each bounding box** (with a small margin) into its category folder. Other categories (pod borer, "other", ...) are dropped. `black_pod` is topped up from the Kaggle "black pod rot" folder.

In [ ]:
import json, os, shutil, glob, cv2

ROOT = '/content'
DATA = f'{ROOT}/data'
os.makedirs(DATA, exist_ok=True)

TARGETS = {'anthracnose', 'black_pod', 'cssvd', 'healthy'}

# Map annotation category names -> our slugs. Print what you actually see and
# adjust this dict if the names differ (e.g. 'pod rot', 'Cocoa swollen shoot').
NAME_MAP = {
    'cssvd': 'cssvd', 'cocoa swollen shoot virus': 'cssvd', 'swollen shoot': 'cssvd',
    'anthracnose': 'anthracnose',
    'black pod': 'black_pod', 'black pod rot': 'black_pod', 'pod rot': 'black_pod',
    'healthy': 'healthy',
}
for t in TARGETS:
    os.makedirs(f'{DATA}/{t}', exist_ok=True)

In [ ]:
# --- Crop KaraAgroAI bboxes ---
img_paths = sorted(glob.glob('karaagro/**/*.jpg', recursive=True) + glob.glob('karaagro/**/*.jpeg', recursive=True) + glob.glob('karaagro/**/*.png', recursive=True))
ann_paths = sorted(glob.glob('karaagro/**/*.json', recursive=True))
print(f'{len(img_paths)} images, {len(ann_paths)} annotation files')

def crop_bboxes(ann_path):
    """Return dict image_filename -> list of (class_slug, x1, y1, x2, y2).
    Handles common detection JSON layouts (COCO-style with categories+images, or a simple list)."""
    with open(ann_path) as f:
        raw = json.load(f)
    # COCO style
    if isinstance(raw, dict) and 'images' in raw and 'annotations' in raw and 'categories' in raw:
        cat = {c['id']: str(c.get('name', '')).strip().lower() for c in raw['categories']}
        im = {i['id']: i['file_name'] for i in raw['images']}
        out = {}
        for a in raw['annotations']:
            name = NAME_MAP.get(cat.get(a['category_id'], ''))
            if not name:
                continue
            x, y, w, h = a['bbox']
            out.setdefault(im[a['image_id']], []).append((name, x, y, x + w, y + h))
        return out
    # Simple list: [{image, category, bbox:{x,y,w,h} or [x,y,w,h]}]
    if isinstance(raw, list):
        out = {}
        for a in raw:
            c = str(a.get('category', a.get('class', ''))).strip().lower()
            name = NAME_MAP.get(c)
            if not name:
                continue
            b = a.get('bbox', a.get('box'))
            if isinstance(b, dict):
                x, y = b['x'], b['y']; w, h = b['w'], b['h']
            else:
                x, y, w, h = b
            out.setdefault(a['image'], []).append((name, x, y, x + w, y + h))
        return out
    return {}

saved = {t: 0 for t in TARGETS}
for ann in ann_paths:
    try:
        boxes = crop_bboxes(ann)
    except Exception as e:
        print('skip', ann, e)
        continue
    for fn, entries in boxes.items():
        src = os.path.join(os.path.dirname(ann), fn)
        if not os.path.exists(src):
            src = os.path.join('karaagro', fn)
        if not os.path.exists(src):
            continue
        im = cv2.imread(src)
        if im is None:
            continue
        H, W = im.shape[:2]
        stem = os.path.splitext(os.path.basename(fn))[0]
        for i, (cls, x1, y1, x2, y2) in enumerate(entries):
            # expand bbox by 10%, clamp to image bounds
            mx, my = (x2 - x1) * 0.1, (y2 - y1) * 0.1
            x1, y1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
            x2, y2 = min(W, int(x2 + mx)), min(H, int(y2 + my))
            if x2 - x1 < 8 or y2 - y1 < 8:
                continue
            crop = im[y1:y2, x1:x2]
            cv2.imwrite(f'{DATA}/{cls}/{stem}_{i}.jpg', crop)
            saved[cls] += 1

print('After KaraAgroAI crops:', saved)

In [ ]:
# --- Copy black pod images from Kaggle / GitHub mirror ---
import shutil, glob

bp_sources = glob.glob('kaggle_cacao/**/black*pod*/**', recursive=True) + \
             glob.glob('kaggle_cacao/**/*Black Pod*/**', recursive=True)
bp_files = [f for f in glob.glob('kaggle_cacao/**/*.jpg', recursive=True) + glob.glob('kaggle_cacao/**/*.JPG', recursive=True)
            if 'pod' in os.path.basename(os.path.dirname(f)).lower() and 'black' in os.path.basename(os.path.dirname(f)).lower()]
for i, f in enumerate(bp_files):
    shutil.copy(f, f'{DATA}/black_pod/kaggle_{i}.jpg')

n = len(glob.glob(f'{DATA}/black_pod/*.jpg'))
print(f'black_pod total: {n} images')
if n < 200:
    print('!! black_pod is thin — oversampling/augmentation below compensates, but add more data if you can.')

In [ ]:
import glob
for t in sorted(TARGETS):
    print(t, len(glob.glob(f'{DATA}/{t}/*.jpg')))

# If any class is empty, STOP and fix the NAME_MAP / file discovery before training.

## 3. Balance + stratified 80/10/10 split

In [ ]:
import os, random, shutil
from sklearn.model_selection import train_test_split

CAP = 2000  # per-class cap; oversample the minority classes with augmentation during training

for cls in TARGETS:
    files = sorted(glob.glob(f'{DATA}/{cls}/*.jpg'))
    random.Random(42).shuffle(files)
    files = files[:CAP]
    train_f, tmp = train_test_split(files, test_size=0.2, random_state=42)
    val_f, test_f = train_test_split(tmp, test_size=0.5, random_state=42)
    for split, lst in (('train', train_f), ('val', val_f), ('test', test_f)):
        dst = f'{DATA}/{split}/{cls}'
        os.makedirs(dst, exist_ok=True)
        for f in lst:
            shutil.copy(f, os.path.join(dst, os.path.basename(f)))
    print(cls, 'train', len(train_f), 'val', len(val_f), 'test', len(test_f))

## 4. Train MobileNetV3-small

**The `Rescaling(1./127.5, offset=-1)` first layer is the preprocessing contract** — the browser runs raw pixels. Data augmentation only (no rescaling) in `ImageDataGenerator`.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np, os

IMG = 224
BATCH = 32
EPOCHS = 12

train_ds = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.15, horizontal_flip=True,
    brightness_range=[0.8, 1.2], fill_mode='nearest',  # NO rescale here
).flow_from_directory(f'{DATA}/train', target_size=(IMG, IMG), batch_size=BATCH, class_mode='categorical')

val_ds = ImageDataGenerator().flow_from_directory(
    f'{DATA}/val', target_size=(IMG, IMG), batch_size=BATCH, class_mode='categorical')

print('Class indices (order MUST match lib/model.ts CLASS_LABELS):', train_ds.class_indices)
assert list(train_ds.class_indices.keys()) == ['anthracnose', 'black_pod', 'cssvd', 'healthy'], \
    'Class order mismatch — fix folder names/CLASS_LABELS.'

classes = sorted(os.listdir(f'{DATA}/train'))
class_weight = dict(enumerate(compute_class_weight(
    'balanced', classes=classes, y=train_ds.classes)))
print('Class weights:', class_weight)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG, IMG, 3)),
    tf.keras.layers.Rescaling(1. / 127.5, offset=-1),          # PREPROCESSING BAKED IN
    MobileNetV3Small(weights='imagenet', input_shape=(IMG, IMG, 3), include_top=False),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(4, activation='softmax'),
])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
          callbacks=callbacks, class_weight=class_weight)

In [ ]:
# --- Fine-tune: unfreeze the top ~40 layers of the backbone ---
model.layers[2].trainable = True
for layer in model.layers[2].layers[:120]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks, class_weight=class_weight)

In [ ]:
# --- Quick test-set sanity check ---
test_ds = ImageDataGenerator().flow_from_directory(
    f'{DATA}/test', target_size=(IMG, IMG), batch_size=BATCH, class_mode='categorical')
loss, acc = model.evaluate(test_ds, verbose=1)
print(f'TEST accuracy: {acc:.3f}')
if acc < 0.80:
    print('!! Below 0.80 — swap MobileNetV3Small -> EfficientNetV2B0, add data, or train longer.')

## 5. Export to TF.js (float16)

Download `web_model.zip`, extract, and copy `model.json` + `*.bin` into `public/models/` in the Next.js repo.

In [ ]:
model.save('/content/saved_model')

!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model \
    --quantize_float16 /content/saved_model /content/web_model 2>&1 | tail -5

!zip -r /content/web_model.zip /content/web_model
!ls -lh /content/web_model/
!du -sh /content/web_model

from google.colab import files
files.download('/content/web_model.zip')
print('\nPlace web_model.zip contents into the repo at public/models/ (model.json + *.bin).')
print('Verify lib/model.ts CLASS_LABELS ==', list(train_ds.class_indices.keys()))